# Análisis Exploratorio de datos
---

**Autor:** Jaime Lozano Cillero

## 1. Descripción del dataset

El conjunto de datos suministrado contiene información sobre reservas de hoteles hechas a lo largo del tiempo, incluyendo detalles sobre los clientes, el comportamiento de reserva y la probabilidad de cancelación.

El objetivo será predecir si una reserva será cancelada (`is_canceled = 1`) o no (`is_canceled = 0`) aplicando modelos de clasificación binaria.


### Descripción de variables

| Nombre Variable                  | Descripción                                              |
| -------------------------------- | -------------------------------------------------------- |
| `hotel`                          | Tipo de hotel: City Hotel o Resort Hotel                 |
| `is_canceled`                    | Variable objetivo: 1 si fue cancelado, 0 si no           |
| `lead_time`                      | Días entre la reserva y la fecha de llegada              |
| `arrival_date_year`              | Año de llegada                                           |
| `arrival_date_month`             | Mes de llegada                                           |
| `arrival_date_week_number`       | Número de la semana del año                              |
| `arrival_date_day_of_month`      | Día del mes de llegada                                   |
| `stays_in_weekend_nights`        | Noches de fin de semana reservadas                       |
| `stays_in_week_nights`           | Noches entre semana reservadas                           |
| `adults`                         | Número de adultos                                        |
| `children`                       | Número de niños                                          |
| `babies`                         | Número de bebés                                          |
| `meal`                           | Tipo de comida reservada                                 |
| `country`                        | País de origen del cliente                               |
| `market_segment`                 | Canal de marketing (online, offline, grupos...)          |
| `distribution_channel`           | Canal de distribución (directo, TA/TO...)                |
| `is_repeated_guest`              | 1 si el cliente ha estado anteriormente                  |
| `previous_cancellations`         | Nº de cancelaciones anteriores                           |
| `previous_bookings_not_canceled` | Nº de reservas previas no canceladas                     |
| `reserved_room_type`             | Tipo de habitación reservada                             |
| `assigned_room_type`             | Tipo de habitación asignada                              |
| `booking_changes`                | Nº de cambios en la reserva                              |
| `deposit_type`                   | Tipo de depósito: No Deposit, Refundable, etc.           |
| `agent`                          | ID del agente (puede ser nulo)                           |
| `company`                        | ID de la empresa (puede ser nulo)                        |
| `days_in_waiting_list`           | Días en lista de espera                                  |
| `customer_type`                  | Tipo de cliente: Transient, Group, etc.                  |
| `adr`                            | Average Daily Rate (precio promedio por noche)           |
| `required_car_parking_spaces`    | Plazas de parking solicitadas                            |
| `total_of_special_requests`      | Nº de peticiones especiales                              |
| `reservation_status`             | Estado final de la reserva: Check-Out, Canceled, No-Show |
| `reservation_status_date`        | Fecha en que se actualizó el estado                      |

## 2. Configuración del Notebook

### Importación de librerías

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

### Definición de constantes

In [ ]:
PATH_DIRECTORIO_DATOS = "../../data"
PATH_DATASET_RAW = f"{PATH_DIRECTORIO_DATOS}/raw/dataset_practica_final.csv"
PATH_DATASET_PROCESSED_REVISADO = f"{PATH_DIRECTORIO_DATOS}/processed/dataset_practica_final_preprocessed_revisado.csv"

## 3. Carga de datos y observaciones iniciales

In [ ]:
# Cargamos el dataset de la práctica final
df = pd.read_csv(PATH_DATASET_RAW)

# Mostrar la información del dataframe
df.info()

In [ ]:
df.head(10)

In [ ]:
print(f"Dimensiones del dataframe:")
print(f"{df.shape[0]} filas")
print(f"{df.shape[1]} columnas")

In [ ]:
# Fijamos la columna que representa las clases a predecir
target_column = 'is_canceled'

# variable objetivo: is_canceled
df[target_column].value_counts(dropna=False)

In [ ]:
# Obtener las frecuencias absolutas de la columna target
df[target_column].value_counts(normalize=True)

In [ ]:
print(f"\nValores nulos:")
print(f"{df.isna().sum()}")

Columnas con valores nulos:
- ``children``: imputaremos el valor cero.
- ``country``: imputaremos valor 'unknown'.
- ``agent`` y ``company``: el enunciado del proyecto indica que las ausencias de valor no válidas. Son categóricas, las transformaremos a string e imputaremos '0' a los nulos.

In [ ]:
print(f"Filas duplicadas:")
print(f"{df.duplicated().sum()}")


In [ ]:
# Descripción estadística de las columnas numéricas
df.describe(include='number').transpose()

In [ ]:
# Descripción estadística de las columnas string (object)
df.describe(include='object').transpose()

## Preprocesamiento de datos

In [ ]:
# Duplicamos el DataFrame para no modificar el original
df_preprocessed = df.copy()

### Descarte de variables

Se considera que se quiere predecir antes de que se consume el hecho de un check-in o que el cliente no se presente. Por eso, se tendrán en cuenta las variables disponibles en el momento de hacer una predicción.

Las variables descartadas son:

- ``reservation_status`` y ``reservation_status_date`` hacen referencia al estado final de la reserva y no se conocerá cuando se quiera hacer una predicción. Además, ``reservation_status`` constituye una **fuga de datos** hacia el conjunto de entrenamiento porque permite predecir la variable objetivo: Canceled y No-Show vienen a ser is_canceled = 1.
- ``assigned_room_type`` se conoce cuando en un momento posterior al de predicción y además constituye **fuga de datos**. Si está asignada un tipo de habitación significa que la reserva no se canceló.

In [ ]:
discarded_columns = ['assigned_room_type', 'reservation_status', 'reservation_status_date']
df_preprocessed = df_preprocessed.drop(columns=discarded_columns)

### Limpieza y transformación

In [ ]:
# 'agent' y 'company': cambiamos nulos por ceros
for col in ['agent', 'company']:
    df_preprocessed[col] = df_preprocessed[col].fillna(0).astype(int)

# 'children': sustituir el valor nulo por cero.
df_preprocessed['children'] = df_preprocessed['children'].fillna(0).astype(int)

# 'country': sustituir nulos por el valor 'unknown'.
df_preprocessed['country'] = df_preprocessed['country'].fillna('unknown')

# 'arrival_date_month': sustituir valor textual por ordinal.
df_preprocessed['arrival_date_month'] = df_preprocessed['arrival_date_month'].replace({
    'January': '1',
    'February': '2',
    'March': '3',
    'April': '4',
    'May': '5',
    'June': '6',
    'July': '7',
    'August': '8',
    'September': '9',
    'October': '10',
    'November': '11',
    'December': '12'
}).astype(int)


In [ ]:
df_preprocessed.info()

In [ ]:
df_preprocessed.isna().sum()

In [ ]:
pairplot_column_list = [
    'is_canceled',
    'lead_time',
    'stays_in_weekend_nights',
    'stays_in_week_nights',
    'adults',
    'children',
    'babies',
    'is_repeated_guest',
    'previous_cancellations',
    'previous_bookings_not_canceled',
    'booking_changes',
    'agent',
    'company',
    'days_in_waiting_list',
    'adr',
    'required_car_parking_spaces',
    'total_of_special_requests'
]
sns.pairplot(df_preprocessed[pairplot_column_list].sample(n=1000, random_state=42), hue='is_canceled', palette='Set1')

In [ ]:
df_preprocessed.to_csv(PATH_DATASET_PROCESSED_REVISADO, index=False)